In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ollama
from utils import downsample, squashOneHots, cleanWhitespace

In [8]:
def write_prompt(X_train,y_train,X_test,header,filename="prompt.txt"):
    output_string = ""

    output_string += header + "\n\n"

    output_string += "Training data (X_train | y_train):\n"

    train_full = X_train
    train_full["class"] = y_train

    output_string += train_full.to_string() + "\n\n"

    # print augmented matrix
    output_string += "The unlabeled data (X_test) are listed below. Each of them is missing the 'class' column. Your task will be to fill in that column. \n\n"
    
    output_string += X_test.to_string() + "\n\n"

    output_string += "Please proceed in generating a comma-separated list of classes (the rightmost column of the labeled data points, titled 'class') corresponding to each entry in the Unlabeled Data, in order. Your output should be in the format 1,5,2,6,3,5,7,..."
    output_string += f"\nYou should output a total of {len(X_test)} comma-separated classes in the range [{1},{7}]."
    # note: this does not handle the actual input/response with the LLM! Just writes the prompt.

    filename = f"./prompts/{filename}"
    file = open(filename,"w")
    file.write(output_string)
    file.close()
    cleanWhitespace(filename)


In [14]:
train,valid,test = pd.read_csv("data/train.csv"),pd.read_csv("data/valid.csv"),pd.read_csv("data/test.csv")

oneHotColumns = [f"Soil_Type{i}" for i in range(1, 40+1)]

train = squashOneHots(downsample(train, N=25))
valid = squashOneHots(downsample(valid, N=5))
test = squashOneHots(downsample(test, N=5))

X_train, y_train = train.drop(columns="class"), train["class"]
X_valid, y_valid = valid.drop(columns="class"), valid["class"]
X_test, y_test = test.drop(columns="class"), test["class"]

In [15]:
header = "You are tasked with classifying trees into one of seven classes based on several geographic parameters, including elevation, aspect, slope, distance to hydrology, distance to roadways, and hillshade at various times of the day. Your task is to learn associations between these attributes and the seven possible output classes, and then try your best to assign class labels (y_test) to each of the unlabeled entries in the dataset (X_test)."

write_prompt(X_train,y_train,X_test,header)

In [13]:
test_prompt = open("./prompts/prompt.txt","r").read()

stream = ollama.chat(
    model='qwen3:14b',
    messages=[{'role': 'user', 'content': test_prompt}],
    stream=True,
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)


The provided data does not include the "Unlabeled Data" required to generate the requested output. The labeled data shown contains only 17 entries, and no unlabeled data is available for prediction. As a result, it is impossible to produce a valid list of 140 classes corresponding to unlabeled entries. Please provide the unlabeled data to proceed.

In [15]:
# New strategy: Set up a CONVERSING system. Feed the entire training set up-front, then feed it one test point at a time and ask for a label.